# Some True Measures in QMCJu

Translated from QMCPy's some_true_measures.ipynb

Explores the true measures available in QMCJu: Uniform, Gaussian,
BrownianMotion, StudentT, Triangular, Kumaraswamy, JohnsonsSU,
and BernoulliCont. Each transforms uniform [0,1)^d samples into
the target distribution via the inverse CDF (quantile) transform.

In [ ]:
using QMCJu
using Statistics
using Printf

n = 10_000  # samples for empirical statistics

Uniform

In [ ]:
println("="^60)
println("Uniform Measure")
println("="^60)

dd = IIDStdUniform(2; seed=7)
tm = Uniform(dd; lower_bound=-2.0, upper_bound=3.0)
x = gen_samples(dd, n)
xt = transform(tm, x)
@printf("  Range: [%.1f, %.1f], Empirical mean: %.3f (exact: 0.5)\n",
        -2.0, 3.0, mean(xt))
println()

Gaussian

In [ ]:
println("="^60)
println("Gaussian Measure")
println("="^60)

dd = IIDStdUniform(2; seed=7)
tm = Gaussian(dd; mean=0.0, covariance=1.0)
x = gen_samples(dd, n)
xt = transform(tm, x)
@printf("  Standard Normal: mean = %.3f, std = %.3f\n", mean(xt), std(xt))

Custom covariance

In [ ]:
Σ = [4.0 1.0; 1.0 2.0]
tm2 = Gaussian(dd; mean=[1.0, -1.0], covariance=Σ)
xt2 = transform(tm2, gen_samples(IIDStdUniform(2; seed=7), n))
@printf("  Custom: mean₁ = %.2f, mean₂ = %.2f (exact: 1, -1)\n",
        mean(xt2[:, 1]), mean(xt2[:, 2]))
println()

Student-t

In [ ]:
println("="^60)
println("Student-t Measure")
println("="^60)

for df in [2.0, 5.0, 30.0]
    local dd, tm, x, xt
    dd = IIDStdUniform(1; seed=7)
    tm = StudentT(dd; df=df)
    x = gen_samples(dd, n)
    xt = transform(tm, x)
    theoretical_var = df > 2 ? df / (df - 2) : Inf
    @printf("  df = %4.0f: mean = %+.3f, var = %6.3f (exact var = %.3f)\n",
            df, mean(xt), var(xt), theoretical_var)
end
println()

Triangular

In [ ]:
println("="^60)
println("Triangular Measure")
println("="^60)

dd = IIDStdUniform(1; seed=7)
tm = Triangular(dd; lower=0.0, upper=1.0, mode=0.5)
x = gen_samples(dd, n)
xt = transform(tm, x)
exact_mean = (0.0 + 1.0 + 0.5) / 3.0
@printf("  Triangular(0, 1, 0.5): mean = %.4f (exact = %.4f)\n",
        mean(xt), exact_mean)

tm2 = Triangular(dd; lower=-1.0, upper=2.0, mode=1.5)
xt2 = transform(tm2, gen_samples(IIDStdUniform(1; seed=7), n))
exact_mean2 = (-1.0 + 2.0 + 1.5) / 3.0
@printf("  Triangular(-1, 2, 1.5): mean = %.4f (exact = %.4f)\n",
        mean(xt2), exact_mean2)
println()

Kumaraswamy

In [ ]:
println("="^60)
println("Kumaraswamy Measure")
println("="^60)
println("  F⁻¹(u) = (1 - (1-u)^(1/β))^(1/α)")
println()

for (α, β) in [(2.0, 5.0), (0.5, 0.5), (2.0, 2.0)]
    local dd, tm, x, xt
    dd = IIDStdUniform(1; seed=7)
    tm = Kumaraswamy(dd; alpha=α, beta=β)
    x = gen_samples(dd, n)
    xt = transform(tm, x)
    @printf("  α=%.1f, β=%.1f: mean = %.4f, range = [%.4f, %.4f]\n",
            α, β, mean(xt), minimum(xt), maximum(xt))
end
println()

Johnson's SU

In [ ]:
println("="^60)
println("Johnson's SU Measure")
println("="^60)
println("  Transform: z = ξ + λ sinh((Φ⁻¹(u) - γ) / δ)")
println()

dd = IIDStdUniform(1; seed=7)
tm = JohnsonsSU(dd; xi=0.0, lambda=1.0, gamma=0.0, delta=1.0)
x = gen_samples(dd, n)
xt = transform(tm, x)
@printf("  Default (ξ=0, λ=1, γ=0, δ=1): mean = %.3f, std = %.3f\n",
        mean(xt), std(xt))

tm2 = JohnsonsSU(dd; xi=5.0, lambda=2.0, gamma=1.0, delta=0.5)
xt2 = transform(tm2, gen_samples(IIDStdUniform(1; seed=7), n))
@printf("  Custom  (ξ=5, λ=2, γ=1, δ=0.5): mean = %.3f, std = %.3f\n",
        mean(xt2), std(xt2))
println()

Continuous Bernoulli

In [ ]:
println("="^60)
println("Continuous Bernoulli Measure")
println("="^60)

for λ in [0.2, 0.5, 0.8]
    local dd, tm, x, xt
    dd = IIDStdUniform(1; seed=7)
    tm = BernoulliCont(dd; lam=λ)
    x = gen_samples(dd, n)
    xt = transform(tm, x)
    @printf("  λ = %.1f: mean = %.4f, range = [%.4f, %.4f]\n",
            λ, mean(xt), minimum(xt), maximum(xt))
end
println("  (λ = 0.5 is the identity transform: Uniform(0,1))")
println()

Brownian Motion

In [ ]:
println("="^60)
println("Brownian Motion Measure")
println("="^60)

dd = IIDStdUniform(8; seed=7)
tm = BrownianMotion(dd; drift=0.0)
x = gen_samples(dd, n)
paths = transform(tm, x)
@printf("  d=8 timesteps, n=%d paths\n", n)
@printf("  E[W(T)] = %.3f (exact = 0)\n", mean(paths[:, end]))
@printf("  Var[W(T)] = %.3f (exact = 1)\n", var(paths[:, end]))

With drift

In [ ]:
tm_d = BrownianMotion(dd; drift=0.1)
paths_d = transform(tm_d, gen_samples(IIDStdUniform(8; seed=7), n))
@printf("  With drift 0.1: E[W(T)] = %.3f (exact = 0.1)\n", mean(paths_d[:, end]))
println()

println("="^60)
println("True measures demo completed!")